In [ ]:
#import and config
import pandas as pd
import requests
import os

COINS = ["bitcoin", "ethereum"]
VS_CURRENCY = "usd"
DAYS = 365  # more history is better for ML
RAW_DATA_PATH = "../data/raw"

os.makedirs(RAW_DATA_PATH, exist_ok=True)


In [27]:
#Fetch coin info from coingecko api
def fetch_coin_info(coin_id):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}"
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch info for {coin_id}")

    data = response.json()

    return {
        "coin": coin_id,
        "symbol": data["symbol"],
        "name": data["name"],
        "market_cap_rank": data["market_cap_rank"],
        "circulating_supply": data["market_data"]["circulating_supply"],
        "total_supply": data["market_data"]["total_supply"],
        "max_supply": data["market_data"]["max_supply"],
        "ath": data["market_data"]["ath"]["usd"],
        "atl": data["market_data"]["atl"]["usd"],
        "price_change_24h": data["market_data"]["price_change_24h"],
        "price_change_percentage_24h": data["market_data"]["price_change_percentage_24h"],
        "categories": ", ".join(data["categories"]),
        # NEW: sentiment votes
        "bullish_votes_pct": data.get("sentiment_votes_up_percentage"),
        "bearish_votes_pct": data.get("sentiment_votes_down_percentage")
    }


In [28]:
#Fetch historical market data for a coin
def fetch_coin_history(coin_id, days=365):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {
        "vs_currency": "usd",
        "days": days
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(
            f"Failed to fetch history for {coin_id} | "
            f"Status: {response.status_code} | Response: {response.text}"
        )

    data = response.json()

    df = pd.DataFrame({
        "timestamp": [x[0] for x in data["prices"]],
        "price": [x[1] for x in data["prices"]],
        "market_cap": [x[1] for x in data["market_caps"]],
        "volume": [x[1] for x in data["total_volumes"]],
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["coin"] = coin_id

    return df



In [29]:
import time

all_data = []

for coin in COINS:
    print(f"Loading {coin}...")

    history_df = fetch_coin_history(coin, DAYS)
    info_dict = fetch_coin_info(coin)

    for key, value in info_dict.items():
        history_df[key] = value

    all_data.append(history_df)
    time.sleep(1)  # be kind to CoinGecko


Loading bitcoin...
Loading ethereum...


In [30]:
# Combine all coins into one data frame
combined_df = pd.concat(all_data, ignore_index=True)

# Quick sanity check
print(combined_df.shape)  # should be ~730 rows for 2 coins × ~365 days
print(combined_df["coin"].value_counts())
combined_df.head()


(732, 18)
coin
bitcoin     366
ethereum    366
Name: count, dtype: int64


C:\Users\tlili\AppData\Local\Temp\ipykernel_23308\2625190261.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(all_data, ignore_index=True)


,timestamp,price,market_cap,volume,coin,symbol,name,market_cap_rank,circulating_supply,total_supply,max_supply,ath,atl,price_change_24h,price_change_percentage_24h,categories,bullish_votes_pct,bearish_votes_pct
0,2025-02-02,100674.787625,1.996424e+12,2.282778e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
1,2025-02-03,97568.316530,1.933691e+12,5.978423e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
2,2025-02-04,101466.860666,2.011121e+12,1.221640e+11,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
3,2025-02-05,98118.439217,1.943535e+12,7.319669e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
4,2025-02-06,96582.886829,1.912585e+12,4.884896e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38


In [31]:
combined_df.shape


(732, 18)

In [32]:
#save the combined dataframe as csv file 
import os

# Ensure the folder exists
RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

# Generate snapshot file name
snapshot_date = pd.Timestamp.now().strftime("%Y-%m-%d")
file_path = f"{RAW_DATA_PATH}/crypto_market_snapshot_{snapshot_date}.csv"

# Save the CSV
combined_df.to_csv(file_path, index=False)

print(f"✅ Snapshot saved successfully at: {file_path}")


✅ Snapshot saved successfully at: ../data/raw/crypto_market_snapshot_2026-02-01.csv
